In [ ]:
# Refresh all dashboard data and reload DataFrames

import subprocess
import importlib
import sys
import os

# Define script paths
base_dir = r'C:/ai_risa_data'
health_script = os.path.join(base_dir, 'run_ledger_health_summary.py')
recon_script = os.path.join(base_dir, 'run_export_reconciliation_csv.py')
trend_script = os.path.join(base_dir, 'run_reconciliation_trend_report.py')

# Run scripts in sequence
for script in [health_script, recon_script, trend_script]:
    result = subprocess.run([sys.executable, script], capture_output=True, text=True)
    print(f'[{os.path.basename(script)}] Exit code:', result.returncode)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)

# Reload all dashboard DataFrames (assumes reload logic is in a function called reload_dashboard_data)
try:
    reload_dashboard_data()
    print('Dashboard data reloaded.')
except Exception as e:
    print('Error reloading dashboard data:', e)

# Dashboard Data Refresh

Use the cell below to refresh all analytics data for this dashboard. This will:

1. Run the latest ledger health summary
2. Export the latest reconciliation CSV
3. Generate the latest trend report
4. Reload all dashboard dataframes

**No command line required.**

## Interactive Filters

Use the controls below to filter by prediction family, model version, or calibration version.

# AI-RISA Trend Dashboard

This dashboard visualizes reconciliation trends and calibration feedback for AI-RISA.

**Data sources:**
- `reports/reconciliation_trend_report.csv`
- `ledger/ledger_health_snapshot.json`
- `learning/calibration_patch_v1.json`

**Sections:**
1. Import Data
2. Rolling Accuracy and Error Trends
3. Model/Calibration Version Breakdown
4. Underperforming and Improving Families
5. Calibration Patch Recommendations

In [ ]:
# 1. Import Data
import pandas as pd
import matplotlib.pyplot as plt
import json
from pathlib import Path

# File paths
trend_csv = Path('reports/reconciliation_trend_report.csv')
health_json = Path('ledger/ledger_health_snapshot.json')
calib_patch_json = Path('learning/calibration_patch_v1.json')

# Load trend report
trend = pd.read_csv(trend_csv, index_col=0, parse_dates=True) if trend_csv.exists() else pd.DataFrame()

# Load health snapshot
health = json.load(open(health_json)) if health_json.exists() else {}

# Load calibration patch
calib_patch = json.load(open(calib_patch_json)) if calib_patch_json.exists() else {}

print('Trend report shape:', trend.shape)
print('Health snapshot loaded:', bool(health))
print('Calibration patch loaded:', bool(calib_patch))

## 2. Rolling Accuracy and Error Trends

Visualize rolling winner accuracy, method accuracy, and round error from the trend report.

In [ ]:
# Plot rolling accuracy and error trends
if not trend.empty:
    plt.figure(figsize=(10, 4))
    trend['rolling_winner_accuracy'].plot(label='Winner Accuracy')
    trend['rolling_method_accuracy'].plot(label='Method Accuracy')
    plt.ylabel('Accuracy')
    plt.title('Rolling Winner/Method Accuracy')
    plt.legend()
    plt.show()

    plt.figure(figsize=(10, 3))
    trend['rolling_round_error'].plot(label='Round Error', color='orange')
    plt.ylabel('Mean Round Error')
    plt.title('Rolling Mean Round Error')
    plt.legend()
    plt.show()
else:
    print('No trend data available.')

## 3. Model/Calibration Version Breakdown

Show winner accuracy by model version and calibration version from the trend report.

In [ ]:
# Winner accuracy by model/calibration version
import numpy as np

if not trend.empty:
    # These columns may not be in the trend CSV, so reload from reconciliation_export.csv
    recon_csv = Path('exports/reconciliation_export.csv')
    if recon_csv.exists():
        recon = pd.read_csv(recon_csv)
        by_model = recon.groupby('model_version')['winner_correct'].mean()
        by_calib = recon.groupby('calibration_version')['winner_correct'].mean()
        by_model.plot(kind='bar', title='Winner Accuracy by Model Version', ylabel='Accuracy', ylim=(0,1))
        plt.show()
        by_calib.plot(kind='bar', title='Winner Accuracy by Calibration Version', ylabel='Accuracy', ylim=(0,1), color='green')
        plt.show()
    else:
        print('No reconciliation_export.csv available for version breakdown.')
else:
    print('No trend data available.')

## 4. Underperforming and Improving Families

Display the top underperforming and improving prediction families from the trend report.

In [ ]:
# Show underperforming and improving families
if not trend.empty:
    # Reload reconciliation_export.csv for family-level stats
    recon_csv = Path('exports/reconciliation_export.csv')
    if recon_csv.exists():
        recon = pd.read_csv(recon_csv)
        fam = recon.groupby('prediction_family_id')['winner_correct'].agg(['mean', 'count'])
        under = fam.sort_values('mean').head(5)
        over = fam.sort_values('mean', ascending=False).head(5)
        print('Top underperforming families:')
        display(under)
        print('Top improving families:')
        display(over)
    else:
        print('No reconciliation_export.csv available for family breakdown.')
else:
    print('No trend data available.')

## 5. Calibration Patch Recommendations

Show current recommended calibration actions from the latest calibration patch.

In [ ]:
# Show calibration patch recommendations
if calib_patch:
    print('Calibration Patch Generated:', calib_patch.get('generated_timestamp', 'N/A'))
    print('Source reconciliation count:', calib_patch.get('source_reconciliation_count', 'N/A'))
    print('Recommended actions:')
    for action in calib_patch.get('recommended_actions', []):
        print('-', action)
else:
    print('No calibration patch data available.')

In [ ]:
# Interactive filters for prediction_family_id, model_version, calibration_version
import ipywidgets as widgets
from IPython.display import display

# Load reconciliation_export.csv for filtering
recon_csv = Path('exports/reconciliation_export.csv')
recon = pd.read_csv(recon_csv) if recon_csv.exists() else pd.DataFrame()

family_options = sorted(recon['prediction_family_id'].dropna().unique()) if not recon.empty else []
model_options = sorted(recon['model_version'].dropna().unique()) if not recon.empty else []
calib_options = sorted(recon['calibration_version'].dropna().unique()) if not recon.empty else []

family_filter = widgets.Dropdown(options=['(All)'] + family_options, description='Family:')
model_filter = widgets.Dropdown(options=['(All)'] + model_options, description='Model:')
calib_filter = widgets.Dropdown(options=['(All)'] + calib_options, description='Calib:')

filters_box = widgets.HBox([family_filter, model_filter, calib_filter])
display(filters_box)

# Filtered DataFrame

def get_filtered():
    df = recon.copy()
    if family_filter.value != '(All)':
        df = df[df['prediction_family_id'] == family_filter.value]
    if model_filter.value != '(All)':
        df = df[df['model_version'] == model_filter.value]
    if calib_filter.value != '(All)':
        df = df[df['calibration_version'] == calib_filter.value]
    return df

# Display filtered summary
def on_filter_change(change=None):
    df = get_filtered()
    print(f'Filtered rows: {len(df)}')
    if not df.empty:
        print('Winner accuracy:', df["winner_correct"].mean())
        print('Method accuracy:', df["method_correct"].mean())
        print('Mean round error:', df["round_error"].mean())
    else:
        print('No data for selected filters.')

family_filter.observe(on_filter_change, names='value')
model_filter.observe(on_filter_change, names='value')
calib_filter.observe(on_filter_change, names='value')

on_filter_change()